# Denoise a single channel from the EDX using BM3D

In [ ]:
#pip install humanfriendly

In [6]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from EDX import *
from utils import *
from utils_sofima import *
import torch
from torch.utils.data import DataLoader
from model import UNet
from utils_noise import *
from bm3d import bm3d, BM3DStages, bm3d_deblurring
from bm4d import bm4d, BM4DStages
import pickle
import copy
import time
import humanfriendly
from skimage.restoration import estimate_sigma
from sklearn.decomposition import PCA
from skimage.exposure import match_histograms
from pymultiscale import *
import matlab.engine

device = torch.device("mps")
print(device)

%load_ext autoreload
%autoreload 2

mps
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load data (the aligned and spectral-binned tile)

In [2]:
# Load aligned dataset
with open('../preprocessing_basic/results/preprocessed_edx/20251201_142554_tile_aligned.pkl', 'rb') as file:
    tile = pickle.load(file)

#### Load a reference (100 frame tile)

In [3]:
# load data
file_path = "../data/EMD/EDXdataset.emd"

# load and preprocess
edx, haadf, xray_energies = load_EDX(file_path, first_frame=0, last_frame=100, sum_frames=True, haadf_last_frame=False)
print(haadf.shape)

tile_ref = EM_EDX(haadf[0,:,:], edx, xray_energies)
tile_ref.apply("crop", parameters={"crop_idx": (slice(None), slice(None), slice(96, 4096))})
tile_ref.apply("binning", parameters={"dim": (2048, 2048, 250)})
#tile_ref.apply("MeanFilterEDX", parameters={"kernel_size": 3})

# check if same haadf
print(np.array_equal(tile.haadf,tile_ref.haadf))

WARNING | RosettaSciIO | The file contains only one spectrum stream (rsciio.emd._emd_velox:590)
(100, 2048, 2048)
True


### Select a channel and denoise it

In [ ]:
# global sigma
sigma_g = estimate_sigma(np.nan_to_num(tile.EDX))
print(sigma_g)

In [ ]:
sigma_global = False

channel_idx = 3
channel_noisy = tile.EDX[:,:,channel_idx]
channel_noisy = np.nan_to_num(channel_noisy, nan=0.0)
channel_ref = tile_ref.EDX[:,:,channel_idx]
sigma = estimate_sigma(channel_noisy)
print("Sigma estimate: ", sigma)

start = time.perf_counter()
if sigma_global:
    channel_bm3d = bm3d(channel_noisy, sigma_psd=sigma_g, stage_arg=BM3DStages.ALL_STAGES)
else:
    channel_bm3d = bm3d(channel_noisy, sigma_psd=sigma, stage_arg=BM3DStages.ALL_STAGES)

elapsed = time.perf_counter() - start
print("Elapsed time for bm3d all stages:", humanfriendly.format_timespan(elapsed))


In [ ]:
f, ax = plt.subplots(2,3,figsize=(12,6))
ax[0][0].imshow(channel_noisy, cmap='gray'); ax[0][0].set_title('20 frames aligned')
ax[0][1].imshow(channel_bm3d, cmap='gray'); ax[0][1].set_title('20 frames aligned + BM3D')
ax[0][2].imshow(channel_ref, cmap='gray'); ax[0][2].set_title('100 frames unaligned')
ax[1][0].imshow(channel_noisy[500:1000,250:750], cmap='gray')
ax[1][1].imshow(channel_bm3d[500:1000,250:750], cmap='gray')
ax[1][2].imshow(channel_ref[500:1000,250:750], cmap='gray')

plt.suptitle("Denoising channel index: %02d, global sigma estimate: %s" % (channel_idx, sigma_global), fontsize=14)
plt.show()

## BM3D with VST

#### Noise parameters

In [4]:
# select channel and remove padding
h,w,pad_remove = 2048,2048,50
channel_idx = 24
channel_noisy = tile.EDX[pad_remove:h-pad_remove, pad_remove:w-pad_remove,channel_idx]
channel_ref = tile_ref.EDX[pad_remove:h-pad_remove, pad_remove:w-pad_remove,channel_idx]
channel_noisy = np.nan_to_num(channel_noisy, nan=0.0)
channel_noisy = np.ascontiguousarray(channel_noisy)


# start matlab engine and set path
eng = matlab.engine.start_matlab()
matlab_path = eng.genpath('../matlab/')   # add path recursively
eng.addpath(matlab_path, nargout=0)

NameError: name 'matlab' is not defined

#### Estimate paramerters based on the channel itself

In [ ]:
params = eng.estimate_noise(channel_noisy, nargout=1)
print(params)
#eng.quit()

#### Or based on a flattened HSI taken as an image

In [ ]:
pad_remove_x = 50
noisy_HSI_reshaped = tile.EDX[pad_remove_x:h-pad_remove_x, pad_remove_x:w-pad_remove_x,:].reshape(h-2*pad_remove_x,(w-2*pad_remove_x)*250)
print(noisy_HSI_reshaped.shape)
params = eng.estimate_noise(np.ascontiguousarray(noisy_HSI_reshaped), nargout=1)
print(params)
sigma_global = True

#### Or based on the first principal component

In [ ]:
pad_remove_x = 50
new_dim = h-2*pad_remove_x
noisy_HSI_reshaped = tile.EDX[pad_remove_x:h-pad_remove_x, pad_remove_x:w-pad_remove_x,:].reshape((new_dim**2,-1))
pca_1 = PCA(n_components=1)
noisy_HSI_pc0 = pca_1.fit_transform(noisy_HSI_reshaped)
noisy_HSI_pc0 = noisy_HSI_pc0.reshape((new_dim,new_dim))
params = eng.estimate_noise(np.ascontiguousarray(noisy_HSI_pc0), nargout=1)
print(params)
sigma_global = 'PCA'

In [ ]:
f, ax = plt.subplots(figsize=(20,5))
ax.plot([i for i in range(250)],np.abs(pca_1.components_)[0,:],'-or',linewidth=0.5); 
ax.set_title('PC 1 coefficients (abs)'); ax.set_ylabel('Abs. coeff'); ax.set_xlabel('EDX channel index')
plt.grid('ON')
plt.show()

#### Get a and b and perform the VST (variable stabilizing transform)

In [ ]:
scale_factor = 1 #np.abs(pca_1.components_)[0,channel_idx]
a = params[0][0]*scale_factor if params[0][0]>=0 else 1e-30
b = params[0][1]*scale_factor if params[0][1]>=0 else 1e-30


print(a, b)
sigma = np.sqrt(b)

# forward vst
#fz = eng.GenAnscombe_forward(channel_noisy, sigma.astype(channel_noisy.dtype), np.asanyarray(a,dtype=channel_noisy.dtype))
fz = generalized_anscombe(channel_noisy, a, sigma, gain=1.0)

# display 
f, ax = plt.subplots(1,2,figsize=(10,6))
ax[0].imshow(channel_noisy,cmap='gray'); ax[0].set_title('Before')
ax[1].imshow(fz,cmap='gray'); ax[1].set_title('After VST')
plt.show()

#### Scale

In [ ]:
sigma_den = 1
fz, min, max = MinMax(fz, return_extra=True)
sigma_den = sigma_den/(max-min)

#### Perform BM3D then reverse the VST

In [ ]:
# BM3D
channel_bm3d = bm3d(fz, sigma_psd=sigma_den, stage_arg=BM3DStages.ALL_STAGES)
print('finished denoising')

In [ ]:
# optional deblurring
v = np.ones((9, 9))
v = v / np.sum(v)
channel_bm3d = bm3d_deblurring(channel_bm3d, sigma_den, v)
print('finished deblurring')

In [ ]:
# Inverse VST
channel_bm3d = channel_bm3d*(max-min)+min
#channel_bm3d = eng.GenAnscombe_inverse_exact_unbiased(channel_bm3d.astype('float32'), sigma, a);    # Matlab
channel_bm3d = inverse_generalized_anscombe(channel_bm3d, a, sigma, gain=1.0)
channel_bm3d = np.asarray(channel_bm3d)
#channel_bm3d = match_histograms(channel_bm3d, channel_noisy)

#### Display

In [ ]:
f, ax = plt.subplots(2,3,figsize=(15,10))
ax[0][0].imshow(channel_noisy, cmap='gray'); ax[0][0].set_title('20 frames aligned')
ax[0][1].imshow(channel_bm3d, cmap='gray'); ax[0][1].set_title('20 frames aligned + BM3D + VST')
ax[0][2].imshow(channel_ref, cmap='gray'); ax[0][2].set_title('100 frames unaligned')
ax[1][0].imshow(channel_noisy[250:1000,250:1000], cmap='gray')
ax[1][1].imshow(channel_bm3d[250:1000,250:1000], cmap='gray')
ax[1][2].imshow(channel_ref[250:1000,250:1000], cmap='gray')

plt.suptitle("Denoising channel index: %02d, global sigma estimate: %s" % (channel_idx, sigma_global), fontsize=14)
plt.show()

# Remarks:
- Both with and without VST, the outcome highly depends on sigma and is not stable. It works well on some channels and not so well on others

### Show a/b estimates per channel

In [ ]:
params_list = []
for channel_idx in range(tile.EDX_dim[2]):
    channel_noisy = tile.EDX[:,:,channel_idx]
    channel_noisy = np.nan_to_num(channel_noisy, nan=0.0)
    h,w,pad_remove = 2048,2048,50
    channel_noisy = channel_noisy[pad_remove:h-pad_remove, pad_remove:w-pad_remove]
    channel_noisy = np.ascontiguousarray(channel_noisy)
    params = eng.estimate_noise(channel_noisy, nargout=1)
    a = params[0][0] 
    b = params[0][1] 
    params_list.append([a,b])
    print("Channel: %03d" % channel_idx, end='\r') 

In [ ]:
f, ax  = plt.subplots(figsize=(15,10))
ax.plot([i for i in range(250)], [i[0] for i in params_list], '-*b')
ax.plot([i for i in range(250)], [i[1] for i in params_list], '-or')

In [ ]:
a = np.mean([i[0] for i in params_list])
b = np.mean([i[1] for i in params_list])
print(a,b)